# INF01090 - Ciência de Dados

# Lab Task 01 — Visualization Techniques with Altair using your own dataset (4 people)

Choose an interesting dataset to explore and produce visualizations in a similar fashion described in the preparation notebook. Visualization is not only about coding, it is about making **good analytical and design choices**.

Deliver the results as a notebook, which can run in other machines. Include the dataset with the submission if it can not be downloaded from a repository. 

## Goals

- choose an interesting dataset
- adapt the ideas described in the preparatory notebook to your dataset
- explain the results and interesting visualizations obtained from your dataset 
- feel free to add more advanced analysis and algorithms, that help better understand the dataset
- use vibe coding extensively (when it makes sense) to improve the final results
- **deliver a notebook with the solution using Moodle** (include your dataset if not able to download directly from the notebook) 

## Awards
- the top 3 ranked solutions will be asked to demonstrate the results in a subsequent lab class

## 1. Choose your own dataset 

### Good dataset criteria

Choose a dataset that is:
- tabular
- not too large
- understandable
- rich enough to support multiple questions

It should ideally contain:
- at least one numerical variable
- at least one categorical variable
- optionally a time variable

### Examples

- sports statistics
- movies or streaming data
- flights
- public health data
- environmental data
- education data

### Checklist for your own dataset

Before creating charts, answer these questions:

1. What is one interesting question I want to answer?
2. Which columns are numerical?
3. Which columns are categorical?
4. Is there a time column?
5. Are there missing values?
6. Which chart type best matches the question?
7. What should the viewer learn from the chart?

## 2. Load and inspect the data



In [42]:
!pip install pandas altair vega_datasets scikit-learn matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----------------------------- ---------- 6.0/8.3 MB 50.7 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.3 MB 16.9 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 14.1 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 76.9 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)

   ---------------------------------------- 0/6 [pyparsing]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# Importando as bibliotecas necessárias (conforme o Lab Preparation)
import pandas as pd
import altair as alt

# Desabilitando o limite de linhas para evitar erros com datasets maiores
alt.data_transformers.disable_max_rows()

# 1. Carregando os dados
# Substitua 'foodpanda.csv' pelo nome exato do arquivo que vocês baixaram
df = pd.read_csv('foodpanda.csv')

# 2. Inspecionando o dataset (Respondendo ao checklist da Etapa 1)
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

print("--- Informações das Colunas e Tipos de Dados ---")
df.info()

print("\n--- Valores Nulos por Coluna ---")
print(df.isna().sum())

# Mostrando as 5 primeiras linhas para entender a "cara" dos dados
df.head()

## 2.1 Processamento e Geração das Amostras

In [43]:
import pandas as pd
import altair as alt
import numpy as np

# Desabilitar max rows
alt.data_transformers.disable_max_rows()

# =====================================================================
# 1. GERANDO AS 10 AMOSTRAS (Literatura Acadêmica) - ~600 linhas cada
# =====================================================================

# 01. População (Baseline)
df_pop = df.copy()
df_pop['Método'] = '01. População (6000)'

# 02. Aleatória Simples (Simple Random)
df_simples = df.sample(n=600, random_state=42).copy()
df_simples['Método'] = '02. Aleatória Simples'

# 03. Estratificada (Stratified) - Mantém a proporção por cidade
df_estratificada = df.groupby('city', group_keys=False).apply(lambda x: x.sample(frac=0.1, random_state=42)).copy()
df_estratificada['Método'] = '03. Estratificada (Cidade)'

# 04. Sistemática (Systematic) - Pula de 10 em 10 registros
df_sistematica = df.iloc[::10].copy()
df_sistematica['Método'] = '04. Sistemática (Passo 10)'

# 05. Conglomerados (Cluster) - Sorteia 1 cidade e foca nela (Simulando um cluster geográfico)
cidade_sorteada = df['city'].drop_duplicates().sample(1, random_state=42).iloc[0]
# Usamos replace=True apenas por segurança caso a cidade tenha menos de 600 pedidos
df_conglomerado = df[df['city'] == cidade_sorteada].sample(n=600, random_state=42, replace=True).copy()
df_conglomerado['Método'] = '05. Conglomerados (1 Cidade)'

# 06. Múltiplos Estágios (Multi-stage) - Estágio 1: Sorteia 2 cidades. Estágio 2: Amostra simples dentro delas.
cidades_estagio = df['city'].drop_duplicates().sample(2, random_state=99)
df_multi = df[df['city'].isin(cidades_estagio)].sample(n=600, random_state=42).copy()
df_multi['Método'] = '06. Múltiplos Estágios'

# 07. PPS - Probabilidade Proporcional ao Tamanho (Proportional to Size)
# Pedidos mais caros têm maior chance matemática de serem sorteados.
df_pps = df.sample(n=600, weights='price', random_state=42).copy()
df_pps['Método'] = '07. PPS (Peso: Preço)'

# 08. Conveniência (Convenience) - Pega as primeiras 600 linhas (Não-probabilístico)
df_conveniencia = df.head(600).copy()
df_conveniencia['Método'] = '08. Conveniência (Head)'

# 09. Cotas (Quota) - Força 50% Homens e 50% Mulheres, independente da população real
df_cotas = pd.concat([
    df[df['gender'] == 'Male'].sample(n=300, random_state=42, replace=True),
    df[df['gender'] == 'Female'].sample(n=300, random_state=42, replace=True)
])
df_cotas['Método'] = '09. Cotas (50/50 Gênero)'

# 10. Intencional ou Julgamento (Purposive) - Foco específico do pesquisador nos clientes mais assíduos
df_intencional = df.nlargest(600, 'order_frequency').copy()
df_intencional['Método'] = '10. Intencional (Top Frequentes)'

# Juntando tudo no DataFrame de comparação
df_comparacao = pd.concat([
    df_pop, df_simples, df_estratificada, df_sistematica, 
    df_conglomerado, df_multi, df_pps, df_conveniencia, 
    df_cotas, df_intencional
])

# =====================================================================
# 1.5 OUTPUT EM TEXTO: RESUMO DAS AMOSTRAS E SANITY CHECK
# =====================================================================

# Agrupando os dados para mostrar o tamanho de cada amostra e suas médias
resumo_texto = df_comparacao.groupby('Método').agg(
    Qtd_Linhas=('customer_id', 'count'),
    Preço_Médio=('price', 'mean'),
    Fidelidade_Média=('loyalty_points', 'mean')
).round(2) # Arredonda para 2 casas decimais para ficar bonito

# O display() no Jupyter formata a tabela como HTML bonitinho (melhor que print)
display(resumo_texto)

,Qtd_Linhas,Preço_Médio,Fidelidade_Média
Método,,,
01. População (6000),6000,800.52,250.17
02. Aleatória Simples,600,808.38,246.41
03. Estratificada (Cidade),601,798.84,240.06
04. Sistemática (Passo 10),600,812.89,247.74
05. Conglomerados (1 Cidade),600,814.73,244.83
06. Múltiplos Estágios,600,824.22,247.99
07. PPS (Peso: Preço),600,1001.16,241.20
08. Conveniência (Head),600,838.14,246.82
09. Cotas (50/50 Gênero),600,814.30,250.18


## 2.2 Geração do Dashboard Altair

In [40]:
# =====================================================================
# 2. DASHBOARD 2x2 UNIFORMIZADO (ALTAIR)
# =====================================================================

LARGURA = 320
ALTURA = 220

# Eixo Y padrão para os gráficos da esquerda (garante que o texto ocupe o mesmo espaço)
eixo_y_esq = alt.Y('Método:N', title=None, sort='ascending', axis=alt.Axis(labelFontWeight='bold', labelLimit=200))
# Eixo Y invisível para os gráficos da direita (cola os gráficos e evita repetição)
eixo_y_dir = alt.Y('Método:N', title=None, sort='ascending', axis=alt.Axis(labels=False, ticks=False, domain=False))

# --- 1. Boxplot (Preços) ---
c1 = alt.Chart(df_comparacao).mark_boxplot(extent='min-max', size=12).encode(
    x=alt.X('price:Q', title='Preço (Contínuo)'),
    y=eixo_y_esq,
    color=alt.Color('Método:N', legend=None, scale=alt.Scale(scheme='tableau10'))
).properties(
    width=LARGURA, height=ALTURA, title='1. Viés de Distribuição (Preços)'
)

# --- 2. Scatter (Média de Fidelidade) ---
media_real = df_pop['loyalty_points'].mean()
pontos = alt.Chart(df_comparacao).mark_circle(size=90, opacity=1).encode(
    x=alt.X('mean(loyalty_points):Q', title='Média de Pontos', scale=alt.Scale(zero=False)),
    y=eixo_y_dir, # Eixo oculto
    color=alt.Color('Método:N', legend=None)
)
linha = alt.Chart(pd.DataFrame({'m': [media_real]})).mark_rule(color='red', strokeDash=[5,5], strokeWidth=2).encode(x='m:Q')
c2 = (pontos + linha).properties(
    width=LARGURA, height=ALTURA, title='2. Viés de Estimativa (Fidelidade)'
)

# --- 3. Barras (Categorias) ---
c3 = alt.Chart(df_comparacao).mark_bar().encode(
    x=alt.X('count():Q', stack='normalize', title='Proporção (%)', axis=alt.Axis(format='%')),
    y=eixo_y_esq,
    # Legenda jogada para baixo para não espremer a largura do gráfico
    color=alt.Color('category:N', legend=alt.Legend(orient='bottom', title=None), scale=alt.Scale(scheme='set2'))
).properties(
    width=LARGURA, height=ALTURA, title='3. Proporções Multinomiais (Categorias)'
)

# --- 4. Barras (Gênero) ---
c4 = alt.Chart(df_comparacao).mark_bar().encode(
    x=alt.X('count():Q', stack='normalize', title='Proporção (%)', axis=alt.Axis(format='%')),
    y=eixo_y_dir, # Eixo oculto
    color=alt.Color('gender:N', legend=alt.Legend(orient='bottom', title=None), scale=alt.Scale(scheme='pastel2'))
).properties(
    width=LARGURA, height=ALTURA, title='4. A Falácia das Cotas (Gênero)'
)

# --- CONSTRUÇÃO DO PAINEL ---
painel_uniforme = ((c1 | c2) & (c3 | c4)).resolve_scale(
    color='independent'
).configure_view(
    strokeWidth=0 # Tira as bordas
).configure_title(
    fontSize=14, color='#333333', anchor='middle' # Centraliza e padroniza os títulos
)

# Exibe o gráfico final
painel_uniforme

alt.VConcatChart(...)

## 2.3 Escolha Acadêmica do Melhor Método de Amostragem

In [44]:
# =====================================================================
# DEFINIÇÃO DA AMOSTRA FINAL E LIMPEZA
# =====================================================================

# =====================================================================
# 3.3 AVALIAÇÃO MATEMÁTICA E SELEÇÃO ALGORÍTMICA
# =====================================================================
print("\n⚙️ Rodando Algoritmo de Validação Estatística (TVD e MAPE)...")

# 1. Calculando os baselines da População (Verdade Absoluta)
pop_price_mean = df_pop['price'].mean()
pop_loyalty_mean = df_pop['loyalty_points'].mean()
pop_cat_prop = df_pop['category'].value_counts(normalize=True)
pop_gender_prop = df_pop['gender'].value_counts(normalize=True)

resultados = []

# 2. Loop para avaliar cada método de amostragem
for metodo in df_comparacao['Método'].unique():
    if 'População' in metodo: 
        continue # Pula a base original
        
    df_m = df_comparacao[df_comparacao['Método'] == metodo]
    
    # Cálculo do MAPE para variáveis numéricas
    err_price = abs(df_m['price'].mean() - pop_price_mean) / pop_price_mean
    err_loyalty = abs(df_m['loyalty_points'].mean() - pop_loyalty_mean) / pop_loyalty_mean
    
    # Cálculo do TVD para variáveis categóricas (garantindo alinhamento de índices)
    m_cat_prop = df_m['category'].value_counts(normalize=True).reindex(pop_cat_prop.index, fill_value=0)
    tvd_cat = 0.5 * np.sum(np.abs(pop_cat_prop - m_cat_prop))
    
    m_gender_prop = df_m['gender'].value_counts(normalize=True).reindex(pop_gender_prop.index, fill_value=0)
    tvd_gender = 0.5 * np.sum(np.abs(pop_gender_prop - m_gender_prop))
    
    # Score Final: Soma das divergências (Quanto menor, melhor)
    divergencia_total = err_price + err_loyalty + tvd_cat + tvd_gender
    
    resultados.append({
        'Método': metodo,
        'Erro Preço (MAPE)': err_price,
        'Erro Fidelidade (MAPE)': err_loyalty,
        'Erro Categórico (TVD)': tvd_cat,
        'Erro Gênero (TVD)': tvd_gender,
        'Divergência Total': divergencia_total
    })

# 3. Criando o DataFrame de Ranking e ordenando do melhor pro pior
df_ranking = pd.DataFrame(resultados).sort_values('Divergência Total').reset_index(drop=True)

print("🏆 RANKING DOS MÉTODOS DE AMOSTRAGEM (Menor erro vence):")
# Estilizando com background gradient (heatmap) para destacar os piores e melhores erros
display(df_ranking.style.background_gradient(subset=['Divergência Total'], cmap='RdYlGn_r'))

# =====================================================================
# DEFINIÇÃO DINÂMICA DA AMOSTRA FINAL E LIMPEZA
# =====================================================================

# O algoritmo extrai automaticamente o nome do vencedor
metodo_vencedor = df_ranking.iloc[0]['Método']
print(f"\n✅ Seleção Algorítmica Concluída! O método vencedor é: {metodo_vencedor}")

# Salvamos dinamicamente o dataset vencedor como o nosso principal
df_amostra = df_comparacao[df_comparacao['Método'] == metodo_vencedor].drop(columns=['Método']).copy()

print(f"Dimensões do dataset oficial de trabalho: {df_amostra.shape[0]} linhas e {df_amostra.shape[1]} colunas.")

# Limpeza de memória
del df_pop, df_simples, df_estratificada, df_sistematica, df_conglomerado 
del df_multi, df_pps, df_conveniencia, df_cotas, df_intencional, df_comparacao, resultados



⚙️ Rodando Algoritmo de Validação Estatística (TVD e MAPE)...
🏆 RANKING DOS MÉTODOS DE AMOSTRAGEM (Menor erro vence):


,Método,Erro Preço (MAPE),Erro Fidelidade (MAPE),Erro Categórico (TVD),Erro Gênero (TVD),Divergência Total
0,04. Sistemática (Passo 10),0.015445,0.009715,0.044500,0.003667,0.073326
1,06. Múltiplos Estágios,0.029602,0.008729,0.031500,0.005833,0.075664
2,02. Aleatória Simples,0.009815,0.015044,0.040167,0.012833,0.077859
3,03. Estratificada (Cidade),0.002101,0.040414,0.029627,0.031387,0.103529
4,10. Intencional (Top Frequentes),0.000920,0.027886,0.051333,0.029167,0.109306
5,08. Conveniência (Head),0.046987,0.013385,0.038500,0.013000,0.111873
6,05. Conglomerados (1 Cidade),0.017750,0.021366,0.051833,0.053000,0.143949
7,07. PPS (Peso: Preço),0.250631,0.035870,0.037000,0.018833,0.342334
8,09. Cotas (50/50 Gênero),0.017210,0.000032,0.024500,0.327500,0.369242



✅ Seleção Algorítmica Concluída! O método vencedor é: 04. Sistemática (Passo 10)
Dimensões do dataset oficial de trabalho: 600 linhas e 20 colunas.
